In [1]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler

In [2]:
# read in csv
df = pd.read_csv('movies_with_cast_award_totals.csv')

In [3]:
# define feature columns

# Film-level precursor results
precursor_cols = [
    'critics_choice_nom', 'critics_choice_win',
    'bafta_nom', 'bafta_win',
    'golden_globes_nom', 'golden_globes_win',
    'pga_nom', 'pga_win',
    'sag_nom', 'sag_win',
]

# Cast career history
cast_cols = [c for c in df.columns if c.startswith('maj_') or c.startswith('grp_')]

feature_cols = precursor_cols + cast_cols
target_col = 'oscar_win'

In [4]:
# feature engineering for whole csv
def add_engineered_features(df):
    d = df.copy()
    
    # Win rates for each precursor
    for show in ['critics_choice', 'bafta', 'golden_globes', 'pga', 'sag']:
        nom = f'{show}_nom'
        win = f'{show}_win'
        d[f'{show}_winrate'] = np.where(d[nom] > 0, d[win] / d[nom], 0)
    
    # Combine cast awards for grand totals
    oscar_cols  = [c for c in d.columns if 'academy_awards' in c]
    bafta_cols  = [c for c in d.columns if 'bafta' in c and c.startswith('maj_')]
    actor_cols  = [c for c in d.columns if 'actor_awards' in c]
    dga_cols    = [c for c in d.columns if 'directors_guild' in c]
    
    d['cast_oscar_wins_total']  = d[[c for c in oscar_cols if '_wins' in c]].sum(axis=1)
    d['cast_oscar_noms_total']  = d[[c for c in oscar_cols if '_noms' in c]].sum(axis=1)
    d['cast_bafta_wins_total']  = d[[c for c in bafta_cols if '_wins' in c]].sum(axis=1)
    d['cast_actor_wins_total']  = d[[c for c in actor_cols if '_wins' in c]].sum(axis=1)
    d['cast_dga_wins_total']    = d[[c for c in dga_cols   if '_wins' in c]].sum(axis=1)
    
    return d

df = add_engineered_features(df)

In [5]:
# Update feature list with engineered columns
winrate_cols    = [c for c in df.columns if c.endswith('_winrate')]
composite_cols  = ['cast_oscar_wins_total', 'cast_oscar_noms_total',
                   'cast_bafta_wins_total', 'cast_actor_wins_total',
                   'cast_dga_wins_total']

# final list of features to train on
feature_cols = precursor_cols + winrate_cols + composite_cols

In [6]:
# Train on all years up to year N, predict year N+1
years = sorted(df['year'].unique())
min_train_years = 20

results = []

# If year after min threshold, predict then add to the training set
for i, test_year in enumerate(years):
    train_years = years[:i]
    if len(train_years) < min_train_years:
        continue
    
    train = df[df['year'].isin(train_years)]
    test  = df[df['year'] == test_year]
    
    X_train = train[feature_cols].fillna(0)
    y_train = train[target_col]
    X_test  = test[feature_cols].fillna(0)
    y_test  = test[target_col]
    
    # Scale features
    scaler = StandardScaler()
    X_train_s = scaler.fit_transform(X_train)
    X_test_s  = scaler.transform(X_test)
    
    # Fit model
    model = LogisticRegression(
        class_weight='balanced',
        max_iter=1000,
        C=1.0,
        solver='lbfgs'
    )
    model.fit(X_train_s, y_train)
    
    # Predict probabilities
    probs = model.predict_proba(X_test_s)[:, 1]
    
    # Key metric: did the model give the actual winner the highest probability?
    test_result = test[['title', target_col]].copy()
    test_result['win_prob'] = probs
    test_result = test_result.sort_values('win_prob', ascending=False)
    
    actual_winner = test_result[test_result[target_col] == 1]['title'].values
    predicted_top = test_result.iloc[0]['title']
    
    correct = len(actual_winner) > 0 and predicted_top in actual_winner
    
    results.append({
        'year': test_year,
        'correct_winner_predicted': correct,
        'predicted': predicted_top,
        'actual': actual_winner[0] if len(actual_winner) > 0 else 'none',
        'winner_rank': test_result[test_result[target_col] == 1].index[0]
                       if len(actual_winner) > 0 else None,
    })
    
    # print the results
    print(f"{test_year} | Predicted: {predicted_top:<40} Actual: {actual_winner[0] if len(actual_winner) > 0 else 'none':<40}")


2016 | Predicted: Spotlight                                Actual: Spotlight                               
2017 | Predicted: Hidden Figures                           Actual: Moonlight                               
2018 | Predicted: The Shape of Water                       Actual: The Shape of Water                      
2019 | Predicted: Black Panther                            Actual: Green Book                              
2020 | Predicted: Parasite                                 Actual: Parasite                                
2021 | Predicted: Nomadland                                Actual: Nomadland                               
2022 | Predicted: CODA                                     Actual: CODA                                    
2023 | Predicted: Everything Everywhere All at Once        Actual: Everything Everywhere All at Once       
2024 | Predicted: Oppenheimer                              Actual: Oppenheimer                             
2025 | Predicted: Anora     

In [7]:
# summary metrics
results_df = pd.DataFrame(results)
accuracy = results_df['correct_winner_predicted'].mean()
print(f"\nWinner correctly predicted: {results_df['correct_winner_predicted'].sum()} / {len(results_df)} years ({accuracy:.1%})")

# find feature importance for the entire dataset
X_all = df[feature_cols].fillna(0)
y_all = df[target_col]
scaler_final = StandardScaler()
X_all_s = scaler_final.fit_transform(X_all)

final_model = LogisticRegression(class_weight='balanced', max_iter=1000, C=1.0)
final_model.fit(X_all_s, y_all)

coef_df = pd.DataFrame({
    'feature': feature_cols,
    'coefficient': final_model.coef_[0]
}).sort_values('coefficient', ascending=False)

print("\nTop positive predictors (higher = more likely to win):")
print(coef_df.head(10).to_string(index=False))

print("\nTop negative predictors:")
print(coef_df.tail(5).to_string(index=False))


Winner correctly predicted: 9 / 11 years (81.8%)

Top positive predictors (higher = more likely to win):
               feature  coefficient
               sag_nom     0.792986
             bafta_nom     0.734469
   cast_dga_wins_total     0.639052
           pga_winrate     0.604974
               pga_win     0.604974
 cast_oscar_wins_total     0.598754
 cast_actor_wins_total     0.583012
    critics_choice_win     0.406705
critics_choice_winrate     0.406705
     golden_globes_nom     0.257753

Top negative predictors:
              feature  coefficient
            bafta_win    -0.116679
cast_bafta_wins_total    -0.163441
              pga_nom    -0.360390
   critics_choice_nom    -0.557019
cast_oscar_noms_total    -1.696279
